# ORACLE-2 embeddings for TabFM

Embeds every BTS source with the ORACLE-2 Lite (light curve) and image-branch (ConvNeXt, Pan-STARRS g/r/i) models, truncated to `days` after trigger, and saves one CSV per split with the ZTFID, labels, metadata, and both latent embeddings. Also prints the depth 1 and depth 2 macro F1 of each model as a sanity check.

In [4]:
import numpy as np
import pandas as pd
import torch

from functools import partial
from pathlib import Path
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm import tqdm

from oracle.architectures import GRU_Improved
from oracle.presets import get_model
from oracle.custom_datasets.BTS import (
    BTS_LC_Dataset, custom_collate_BTS, truncate_BTS_light_curve_by_days_since_trigger,
    BTS_train_parquet_path, BTS_val_parquet_path, BTS_test_parquet_path,
    time_independent_feature_list, meta_data_feature_list, flag_value,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

REPO = Path.cwd().parent
LITE_DIR = REPO / 'models/BTSv2-lite/fancy-elevator-567'
IMAGE_DIR = REPO / 'models/BTSv2_PSonly/charmed-wind-579'
OUT_DIR = Path('embeddings')

SPLITS = {'train': BTS_train_parquet_path, 'val': BTS_val_parquet_path, 'test': BTS_test_parquet_path}
METADATA_COLS = time_independent_feature_list + meta_data_feature_list
F1_DEPTHS = [1, 2]

In [5]:
class BTS_PS_Dataset(BTS_LC_Dataset):
    """BTS dataset with the Pan-STARRS (g, r, i) cutout as the image instead of the ZTF reference stamps."""

    def __getitem__(self, index):
        sample = super().__getitem__(index)
        sample['postage_stamp'] = np.asarray(self.parquet_df['ps'][index], dtype=np.float32).reshape(3, 252, 252)
        return sample


def load_model(model_dir):
    model_choice = pd.read_csv(model_dir / 'train_args.csv')['model'][0]
    model = get_model(model_choice)
    model.load_state_dict(torch.load(model_dir / 'best_model_f1.pth', map_location=device), strict=False)
    return model.to(device).eval()


def predict_from_embedding(model, emb):
    # Mirrors each model's forward() so the image model isn't run twice per batch
    logits = model.fc_out(model.gelu(emb)) if isinstance(model, GRU_Improved) else model.final_out(emb)
    probs = model.taxonomy.get_class_probabilities(model.taxonomy.get_conditional_probabilities(logits))
    return pd.DataFrame(probs.cpu().numpy(), columns=model.taxonomy.get_level_order_traversal())


def print_macro_f1(labels, probs, taxonomy):
    # Classes outside the taxonomy (e.g. Anomaly) stay in the CSV but are left out of the F1
    in_taxonomy = np.isin(labels, taxonomy.get_level_order_traversal())
    if not in_taxonomy.all():
        print(f'  F1 excludes {(~in_taxonomy).sum()} sources with classes outside the taxonomy: {sorted(set(labels[~in_taxonomy]))}')
    paths = taxonomy.get_paths(labels[in_taxonomy])
    nodes_by_depth = taxonomy.get_nodes_by_depth()
    for name, prob_dfs in probs.items():
        prob_df = pd.concat(prob_dfs, ignore_index=True)[in_taxonomy]
        scores = []
        for depth in F1_DEPTHS:
            true = np.array([p[depth] if len(p) > depth else None for p in paths], dtype=object)
            pred = prob_df[nodes_by_depth[depth]].idxmax(axis=1).to_numpy()
            keep = true != None
            scores.append(f'depth {depth} = {f1_score(true[keep], pred[keep], average="macro"):.3f}')
        print(f'  {name} macro F1: ' + ', '.join(scores))


@torch.no_grad()
def embed_split(parquet_path, days, models, batch_size=256):
    dataset = BTS_PS_Dataset(parquet_path, transform=partial(truncate_BTS_light_curve_by_days_since_trigger, d=days))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_BTS)

    dfs, probs = [], {name: [] for name in models}
    for batch in tqdm(loader):
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        # Metadata is taken at the last detection within the window; flagged values go back to NaN
        metadata = batch['static'].cpu().numpy()
        parts = [
            pd.DataFrame({'ZTFID': batch['id'], 'class': batch['label'], 'bts_class': batch['raw_label']}),
            pd.DataFrame(np.where(metadata == flag_value, np.nan, metadata), columns=METADATA_COLS),
        ]
        for name, model in models.items():
            emb = model.get_latent_space_embeddings(batch)
            probs[name].append(predict_from_embedding(model, emb))
            parts.append(pd.DataFrame(emb.cpu().numpy(), columns=[f'{name}_emb_{i}' for i in range(emb.shape[1])]))

        dfs.append(pd.concat(parts, axis=1))

    df = pd.concat(dfs, ignore_index=True)
    print_macro_f1(df['class'].to_numpy(), probs, next(iter(models.values())).taxonomy)
    return df


def save_embeddings(days, splits=SPLITS, out_dir=OUT_DIR, batch_size=256):
    models = {'lite': load_model(LITE_DIR), 'image': load_model(IMAGE_DIR)}
    out_dir.mkdir(parents=True, exist_ok=True)

    for split, parquet_path in splits.items():
        print(f'{split}:')
        df = embed_split(parquet_path, days, models, batch_size)
        df.insert(1, 'split', split)
        df.to_csv(out_dir / f'BTS_{split}_{days}d.csv', index=False)
        print(f'Saved {len(df)} rows to {out_dir / f"BTS_{split}_{days}d.csv"}')

In [ ]:
for i in range(11):
    if i != 10:
        continue
    d = int(2**i)
    print("="*16)
    print(f'Embedding for {d} days:')
    save_embeddings(days=d)

Embedding for 1024 days:
train:
Loading dataset from /Users/vedshah/Documents/Research/NU-Miller/Projects/Hierarchical-VT/data/BTS_public/train_PS_ZTF.parquet

Before transforms and mappings, the dataset contains...
          Class  Counts
0           AGN    2144
1          AGN?      16
2        AM CVn       5
3        BL Lac       1
4        Blazar       2
5         CLAGN       6
6            CV     577
7           CV?     115
8   Cataclysmic     158
9          NLS1       3
10       NLSy1?       3
11          QSO     555
12         QSO?       1
13       SLSN-I      69
14     SLSN-I.5       3
15      SLSN-I?       3
16      SLSN-II      23
17        SN II    1084
18     SN II-SL      10
19   SN II-norm      11
20    SN II-pec       2
21       SN II?      23
22       SN IIL       1
23       SN IIP      76
24       SN IIb     104
25   SN IIb-pec       1
26      SN IIb?       4
27       SN IIn     201
28      SN IIn?       6
29        SN Ia    5110
30   SN Ia-00cx       3
31   SN Ia-03fg 

 19%|█▉        | 9/47 [04:28<18:53, 29.84s/it]